In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

FPS = 25


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "evaluation").is_dir():
            return p
    raise FileNotFoundError("Folder 'evaluation' tidak ditemukan dari folder kerja notebook.")


ROOT = find_root()
GT_PATH = ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
PRED_PATH = ROOT / "outputs" / "submission.csv"   # ganti kalau mau bandingkan file lain


def read(path):
    return pd.read_csv(path)["label"].astype(str).str.strip().str.upper().to_numpy()


gt, pr = read(GT_PATH), read(PRED_PATH)
g = np.flatnonzero(gt != "X")
lo, hi = int(g.min()), int(g.max()) + 1
print(f"ROOT: {ROOT}")
print(f"window {lo}-{hi}  panjang {hi - lo}  (7500 kalau kontigu)")


def sec(f):
    return round((f - lo) / FPS, 1)


def segments(a):
    out, s = [], lo
    for t in range(lo + 1, hi + 1):
        if t == hi or a[t] != a[s]:
            out.append((a[s], s, t))
            s = t
    return out


for name, arr in (("GT", gt), ("PREDIKSI", pr)):
    print(f"\nSEGMEN {name}:")
    for lab, s, e in segments(arr):
        print(f"  {lab}  {sec(s):6}-{sec(e):6} s  ({(e - s) / FPS:.1f} s)")

print("\nSTRETCH SALAH >= 1 detik (gt -> prediksi):")
pairs = list(zip(gt[lo:hi], pr[lo:hi]))
s = 0
for i in range(1, len(pairs) + 1):
    if i == len(pairs) or pairs[i] != pairs[s]:
        if pairs[s][0] != pairs[s][1] and i - s >= FPS:
            print(f"  {pairs[s][0]}->{pairs[s][1]}  {sec(lo + s):6}-{sec(lo + i):6} s  ({(i - s) / FPS:.1f} s)")
        s = i

ROOT: C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg
window 11185-18685  panjang 7500  (7500 kalau kontigu)

SEGMEN GT:
  A     0.0-   6.6 s  (6.6 s)
  H     6.6-  14.7 s  (8.1 s)
  A    14.7-  17.6 s  (2.8 s)
  H    17.6-  59.2 s  (41.6 s)
  A    59.2-  66.6 s  (7.4 s)
  D    66.6-  72.2 s  (5.5 s)
  H    72.2- 100.8 s  (28.6 s)
  A   100.8- 104.0 s  (3.2 s)
  H   104.0- 146.4 s  (42.4 s)
  A   146.4- 166.4 s  (20.0 s)
  H   166.4- 167.2 s  (0.8 s)
  A   167.2- 183.2 s  (16.0 s)
  H   183.2- 184.6 s  (1.3 s)
  A   184.6- 188.2 s  (3.6 s)
  D   188.2- 208.6 s  (20.4 s)
  A   208.6- 244.3 s  (35.7 s)
  H   244.3- 247.8 s  (3.5 s)
  D   247.8- 266.1 s  (18.3 s)
  A   266.1- 273.7 s  (7.6 s)
  D   273.7- 288.2 s  (14.5 s)
  H   288.2- 300.0 s  (11.8 s)

SEGMEN PREDIKSI:
  D     0.0-   0.1 s  (0.1 s)
  A     0.1-  11.4 s  (11.4 s)
  H    11.4-  12.9 s  (1.5 s)
  A    12.9-  17.4 s  (4.4 s)
  H    17.4-  20.3 s  (2.9 s)
  A    20.3-  26.3 s  (6.0 s)
  H    26.3-  50.0 s  (23.7 s)
  A    50.

In [1]:
"""Uji sensitivitas parameter di match penuh. TIDAK memakai ground truth.
Jalankan dari terminal (venv aktif):  python scripts/sensitivity.py
"""
import copy
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "main.py").is_file():
            return p
    raise SystemExit("main.py tidak ditemukan dari folder kerja.")


ROOT = find_root()
TMP = ROOT / "outputs" / "sensitivity"
TMP.mkdir(parents=True, exist_ok=True)
FPS = 25

base_cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))

# (jalur kunci di config, nilai dasar dibaca dari config)
KEYS = [
    ("possession_v2", "decoder", "team_switch_penalty"),
    ("possession_v2", "control", "control_min_s"),
    ("possession_v2", "control", "own_radius_m"),
    ("possession_v2", "control", "own_margin_m"),
    ("possession_v2", "dead_ball", "switch_penalty"),
    ("possession_v2", "dead_ball", "min_dead_s"),
]


def run(name, cfg):
    cfg_path = TMP / f"{name}.yaml"
    out_path = TMP / f"{name}.csv"
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    subprocess.run([sys.executable, "main.py", "--config", str(cfg_path), "--output", str(out_path)],
                   cwd=ROOT, check=True, capture_output=True)
    return pd.read_csv(out_path)["label"].astype(str).str.strip().str.upper().to_numpy()


def stats(lab):
    runs, prev, n_run = [], None, 0
    for x in lab:
        if x in ("H", "A"):
            if x == prev:
                n_run += 1
            else:
                if prev is not None:
                    runs.append(n_run)
                prev, n_run = x, 1
        else:
            if prev is not None:
                runs.append(n_run)
            prev, n_run = None, 0
    if prev is not None:
        runs.append(n_run)
    d = (lab == "D").sum()
    return len(runs), (np.mean(runs) / FPS if runs else 0.0), d / max((lab != "X").sum(), 1)


base = run("base", base_cfg)
n0, m0, d0 = stats(base)
print(f"BASE: possession={n0}  rata-rata={m0:.1f}s  D={d0:.1%}\n")
print(f"{'perubahan':50} {'poss':>5} {'rata(s)':>8} {'D%':>6} {'label berubah':>14}")

graded = base != "X"
for path in KEYS:
    node = base_cfg
    for k in path:
        node = node[k]
    for factor in (0.7, 1.3):
        cfg = copy.deepcopy(base_cfg)
        d = cfg
        for k in path[:-1]:
            d = d[k]
        d[path[-1]] = node * factor
        name = f"{path[-1]}_x{factor}"
        lab = run(name, cfg)
        n, m, dd = stats(lab)
        changed = (lab[graded] != base[graded]).mean()
        print(f"{path[-1] + ' x' + str(factor):50} {n:5d} {m:8.1f} {dd:6.1%} {changed:14.1%}")

BASE: possession=344  rata-rata=11.5s  D=37.9%

perubahan                                           poss  rata(s)     D%  label berubah
team_switch_penalty x0.7                             398      9.9  37.9%           0.6%
team_switch_penalty x1.3                             306     12.9  37.9%           0.3%
control_min_s x0.7                                   344     11.5  37.9%           0.2%
control_min_s x1.3                                   329     12.0  37.9%           0.4%
own_radius_m x0.7                                    281     14.0  37.9%           2.1%
own_radius_m x1.3                                    398      9.9  37.9%           1.5%
own_margin_m x0.7                                    349     11.3  37.9%           0.4%
own_margin_m x1.3                                    340     11.6  37.9%           0.2%
switch_penalty x0.7                                  348     11.3  38.0%           0.5%
switch_penalty x1.3                                  344     11.5  37.8%

In [2]:
"""Dari mana flip H/A pendek berasal? TIDAK memakai ground truth.
Jalankan (venv aktif):  python scripts/diagnosa_flip.py
Angka 'support' = bukti yang mendukung label segmennya sendiri (positif = mendukung).
"""
import os
import runpy
import sys
from pathlib import Path

import numpy as np
import pandas as pd

FPS = 25
PUB = (11185, 18685)   # 5 menit pertama permainan (README); hanya untuk memilih segmen yang dicetak


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "main.py").is_file():
            return p
    raise SystemExit("main.py tidak ditemukan dari folder kerja.")


ROOT = find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
OUT = ROOT / "outputs" / "diag_tmp.csv"

import src.possession_v2 as pv2  # noqa: E402

NEEDED = ("touch", "shape", "shape_weight", "tactical_weight", "tactical",
          "evidence", "episodes", "motion")
CAP = {}
_orig = pv2.two_state_viterbi


def _spy(log_odds, penalty):
    if not CAP:
        loc = sys._getframe(1).f_locals
        for k in NEEDED:
            CAP[k] = loc[k]
    return _orig(log_odds, penalty)


pv2.two_state_viterbi = _spy
old_argv = sys.argv
sys.argv = ["main.py", "--config", "config.yaml", "--output", str(OUT)]
try:
    runpy.run_path(str(ROOT / "main.py"), run_name="__main__")
except SystemExit:
    pass
finally:
    sys.argv = old_argv
    pv2.two_state_viterbi = _orig
if not CAP:
    raise SystemExit("Pipeline controlled_possession tidak terpanggil. Cek decoder.method di config.yaml.")

lab = pd.read_csv(OUT)["label"].astype(str).str.strip().str.upper().to_numpy()
n = len(lab)
comps = {
    "touch": CAP["touch"],
    "shape": CAP["shape_weight"] * CAP["shape"],
    "tactical": CAP["tactical_weight"] * CAP["tactical"].log_odds_home,
}
comps["restart"] = CAP["evidence"] - sum(comps.values())
known = CAP["motion"].ball_known
eps = [e for e in CAP["episodes"] if e.controlled]


def segments(a):
    out, s = [], 0
    for t in range(1, n + 1):
        if t == n or a[t] != a[s]:
            if a[s] in ("H", "A"):
                out.append((a[s], s, t))
            s = t
    return out


def support(l, s, e):
    sg = 1.0 if l == "H" else -1.0
    return {k: sg * float(v[s:e].sum()) for k, v in comps.items()}


S = segments(lab)
dur = np.array([(e - s) / FPS for _, s, e in S])
print(f"Possession (seluruh match): {len(S)}  rata-rata {dur.mean():.1f} s")
labels = ["<1 s", "1-2 s", "2-5 s", "5-10 s", "10-20 s", ">=20 s"]
hist, _ = np.histogram(dur, bins=[0, 1, 2, 5, 10, 20, 1e9])
for lbl, c in zip(labels, hist):
    print(f"  {lbl:>8}: {c}")

short = [(l, s, e) for l, s, e in S if (e - s) / FPS < 2.0]
if short:
    dom, vis = {}, []
    for l, s, e in short:
        sp = support(l, s, e)
        d = max(sp, key=sp.get)
        dom[d] = dom.get(d, 0) + 1
        vis.append(known[s:e].mean())
    print(f"\nSegmen pendek (<2 s): {len(short)} dari {len(S)}")
    print("  komponen yang paling mendukung label segmen:", dom)
    print(f"  bola terlihat, median per segmen: {np.median(vis):.0%}")

print("\nSegmen di window publik (agregat per segmen):")
print(" lab  mulai(s)  dur(s)  bola%   touch  shape  tactical  restart  ctrlH ctrlA")
for l, s, e in S:
    if not (PUB[0] <= s < PUB[1]):
        continue
    sp = support(l, s, e)
    ov = [x for x in eps if x.start < e and x.end > s]
    nh = sum(1 for x in ov if x.team == 0)
    print(f"  {l}  {(s - PUB[0]) / FPS:7.1f} {(e - s) / FPS:7.1f}  {known[s:e].mean():4.0%} "
          f"{sp['touch']:7.0f} {sp['shape']:6.0f} {sp['tactical']:9.0f} {sp['restart']:8.0f} "
          f"{nh:6d} {len(ov) - nh:5d}")

Reading tactical tracking (this requires the full licensed ATD)...
Loaded 199,319 frames; 46 track slots
dead ball on 37.9% of graded frames (128 stoppages, 127 live stretches)
touch episodes: 1,345, controlled: 1,082
team-shape model: 6,386 pseudo-labelled frames; agreement with the detector on the held-out half: 85.0%, 84.1%
Wrote C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg\outputs\diag_tmp.csv
Validated labels: {'X': 40462, 'D': 60222, 'A': 36056, 'H': 62579}
Run the organizer's evaluation/validate_submission.py on this exact file too.
Possession (seluruh match): 344  rata-rata 11.5 s
      <1 s: 25
     1-2 s: 31
     2-5 s: 85
    5-10 s: 78
   10-20 s: 59
    >=20 s: 66

Segmen pendek (<2 s): 56 dari 344
  komponen yang paling mendukung label segmen: {'touch': 49, 'restart': 2, 'shape': 5}
  bola terlihat, median per segmen: 100%

Segmen di window publik (agregat per segmen):
 lab  mulai(s)  dur(s)  bola%   touch  shape  tactical  restart  ctrlH ctrlA
  A      0.1    11.4  100

In [3]:
"""Kurva control_min_s di match penuh. TIDAK memakai ground truth.
Jalankan (venv aktif):  python scripts/sweep_control.py
"""
import copy
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

FPS = 25
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "main.py").is_file())
TMP = ROOT / "outputs" / "sensitivity"
TMP.mkdir(parents=True, exist_ok=True)
base_cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))


def run(name, cfg):
    cp, op = TMP / f"{name}.yaml", TMP / f"{name}.csv"
    cp.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    subprocess.run([sys.executable, "main.py", "--config", str(cp), "--output", str(op)],
                   cwd=ROOT, check=True, capture_output=True)
    return pd.read_csv(op)["label"].astype(str).str.strip().str.upper().to_numpy()


def durations(lab):
    out, prev, n = [], None, 0
    for x in lab:
        if x in ("H", "A") and x == prev:
            n += 1
            continue
        if prev is not None:
            out.append(n)
        prev, n = (x, 1) if x in ("H", "A") else (None, 0)
    if prev is not None:
        out.append(n)
    return np.array(out) / FPS


base = run("cm_base", base_cfg)
graded = base != "X"
print(f"{'control_min_s':>14} {'poss':>5} {'rata(s)':>8} {'<1s':>5} {'<2s':>5} {'label berubah':>14}")
for v in (0.5, 0.6, 0.75, 1.0, 1.25, 1.5, 2.0):
    cfg = copy.deepcopy(base_cfg)
    cfg["possession_v2"]["control"]["control_min_s"] = v
    lab = base if v == 0.5 else run(f"cm_{v}", cfg)
    d = durations(lab)
    ch = (lab[graded] != base[graded]).mean()
    print(f"{v:14.2f} {len(d):5d} {d.mean():8.1f} {(d < 1).sum():5d} {(d < 2).sum():5d} {ch:14.1%}")

 control_min_s  poss  rata(s)   <1s   <2s  label berubah
          0.50   344     11.5    25    56           0.0%
          0.60   331     11.9    21    53           0.3%
          0.75   308     12.8    14    44           0.7%
          1.00   293     13.5     7    36           0.9%
          1.25   279     14.1     7    28           1.2%
          1.50   268     14.7     7    21           1.7%
          2.00   250     15.8     7    13           1.8%
